In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='ee-camcoredatabase')

start_date = '1990-01-01'
end_date = '2025-01-01'

# Define Brazil bounding box
bbox = ee.Geometry.BBox(-94.187, -39.020, 37.062, 18.229)

# Load CHIRPS Daily Precipitation dataset
dataset = (
    ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
    .filterDate(start_date, end_date)
    .filterBounds(bbox)
)

# Select the 'precipitation' band
precipitation = dataset.select('precipitation')

# Function to export each daily precipitation image
def export_image(image):
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    task = ee.batch.Export.image.toDrive(
        image=image.clip(bbox),
        description=f"Precip_{date.getInfo()}",
        folder="CHIRPS_BBOX",  # Change export folder
        scale=5000,  # CHIRPS native resolution
        region=bbox,
        maxPixels=1e13
    )
    task.start()
    print(f"Exporting: Precip_{date.getInfo()}")

# Iterate through each daily image and export
precip_list = precipitation.toList(precipitation.size())
for i in range(precip_list.size().getInfo()):
    image = ee.Image(precip_list.get(i))
    export_image(image)